# 步頻標準差盒需圖.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import pandas as pd
import math
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
def calculate_acceleration(speeds, time):

    min_length = min(len(speeds), len(time))
    speeds = speeds[:min_length]
    time = time[:min_length]
    acceleration = np.diff(speeds) / np.diff(time)
    return list(acceleration)


In [ ]:
def process_file(file_name, df):
    if 'kpts' in df.columns:
        df['kpts'] = df['kpts'].str.replace('"', '').str.strip()

        if df['kpts'].isnull().all() or df['kpts'].str.strip().eq('').all():
            print(f"文件 {file_name} 中的 kpts 列包含無效數據，跳過該文件。")
            return None


        kpts_columns = df['kpts'].str.split(',', expand=True)


        for i in range(kpts_columns.shape[1]):
            kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


        df = df.join(kpts_columns, rsuffix='_kpts')


        for col in kpts_columns.columns:
            df[col] = pd.to_numeric(kpts_columns[col], errors='coerce').fillna(0)


        x16 = df['kpt_46']
        x17 = df['kpt_49']

        step_lengths = [step_x(x16.iloc[i], x17.iloc[i]) for i in range(len(x16))]
        average_step_length = sum(step_lengths) / len(step_lengths) if step_lengths else 0

        print(f"{file_name} 平均步距：{average_step_length:.5f}")


        time = pd.Series(range(len(x16)))


        speeds = [step_lengths[i] / time[i] if time[i] > 0 else 0 for i in range(len(step_lengths))]


        acceleration = calculate_acceleration(speeds, time)


        min_length = min(len(acceleration), len(df))
        df = df.iloc[:min_length]
        acceleration = acceleration[:min_length]


        df['step_length'] = step_lengths[:min_length]
        df['speed'] = speeds[:min_length]
        df['acceleration'] = acceleration

        return df
    else:
        print(f"文件 {file_name} 中沒有 'kpts' 列，跳過該文件。")
        return None


In [ ]:
def process_and_plot():
    category_results = {}


    for category in range(1, 5):
        print(f"請上傳第 {category} 類的CSV文件")
        uploaded_files = files.upload()

        all_data = pd.DataFrame()


        for file_name in uploaded_files.keys():
            df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))
            processed_df = process_file(file_name, df)

            if processed_df is not None:
                processed_df.reset_index(drop=True, inplace=True)
                all_data = pd.concat([all_data, processed_df], ignore_index=True)


        category_results[category] = all_data


    plt.figure(figsize=(10, 6))


    data = [category_results[cat]['acceleration'].dropna() for cat in range(1, 5)]


    plt.boxplot(data, patch_artist=True, boxprops=dict(facecolor='lightblue'), showfliers=True)


    plt.title('Comparison of Accelerations for All Categories', fontsize=16)
    plt.xlabel('Category', fontsize=14)
    plt.ylabel('Acceleration', fontsize=14)
    plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
    plt.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
process_and_plot()
